# Análisis del Transporte Público en Lima (2019-2024)

Evolución de pasajeros por sistema de transporte masivo, impacto COVID-19 y recuperación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})

PALETA = ["#1d3557", "#457b9d", "#a8dadc", "#e63946", "#f1c453"]

## Generación de datos con patrones reales

In [ ]:
np.random.seed(42)
meses = pd.date_range("2019-01", "2024-12", freq="MS")
n = len(meses)

estacionalidad = np.array([
    0.95, 0.88, 1.02, 1.00, 1.01, 0.98,
    1.08, 1.03, 1.02, 1.04, 1.00, 0.92,
] * 6)[:n]

def covid_curva(n_meses, base, piso, inicio_caida=14, duracion_caida=3, velocidad_recup=0.02):
    serie = np.full(n_meses, base, dtype=float)
    for i in range(n_meses):
        if i < inicio_caida:
            continue
        elif i < inicio_caida + duracion_caida:
            frac = (i - inicio_caida) / duracion_caida
            serie[i] = base - (base - piso) * frac
        else:
            meses_recup = i - (inicio_caida + duracion_caida)
            recup = 1 - np.exp(-velocidad_recup * meses_recup)
            serie[i] = piso + (base - piso) * recup
    return serie

sistemas = {
    "Metro Línea 1": covid_curva(n, 10.2e6, 1.8e6, velocidad_recup=0.04),
    "Metropolitano": covid_curva(n, 8.5e6, 1.5e6, velocidad_recup=0.035),
    "Corredor Javier Prado": covid_curva(n, 4.8e6, 0.9e6, velocidad_recup=0.03),
    "Corredor TGA": covid_curva(n, 3.9e6, 0.7e6, velocidad_recup=0.028),
    "Corredor SJL": covid_curva(n, 3.2e6, 0.6e6, velocidad_recup=0.025),
}

filas = []
for sistema, base in sistemas.items():
    ruido = np.random.normal(1, 0.03, n)
    pasajeros = (base * estacionalidad * ruido).astype(int)
    pasajeros = np.maximum(pasajeros, 100_000)
    for fecha, pax in zip(meses, pasajeros):
        filas.append({"fecha": fecha, "sistema": sistema, "pasajeros": pax})

df = pd.DataFrame(filas)
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month

print(f"Registros: {len(df)}, Sistemas: {df['sistema'].nunique()}")
df.head(10)

## Resumen por sistema y año

In [ ]:
resumen = df.pivot_table(
    index="sistema", columns="anio",
    values="pasajeros", aggfunc="sum",
)
resumen = (resumen / 1e6).round(1)
resumen.columns = [f"{c} (M)" for c in resumen.columns]
resumen

## Evolución mensual por sistema (área apilada)

In [ ]:
pivote = df.pivot_table(index="fecha", columns="sistema", values="pasajeros")
orden = ["Metro Línea 1", "Metropolitano", "Corredor Javier Prado", "Corredor TGA", "Corredor SJL"]
pivote = pivote[orden]

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(
    pivote.index, *[pivote[col] / 1e6 for col in pivote.columns],
    labels=pivote.columns, colors=PALETA, alpha=0.85,
)
ax.set_ylabel("Pasajeros (millones)")
ax.set_title("Pasajeros Mensuales por Sistema de Transporte")
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-07-01"),
           alpha=0.15, color="red", label="Cuarentena")
plt.tight_layout()
plt.show()

## Participación de mercado: 2019 vs 2023

In [ ]:
comparacion = df[df["anio"].isin([2019, 2023])].copy()
totales = comparacion.groupby(["anio", "sistema"])["pasajeros"].sum().reset_index()
total_anio = totales.groupby("anio")["pasajeros"].transform("sum")
totales["participacion"] = (totales["pasajeros"] / total_anio * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=totales, x="sistema", y="participacion",
    hue="anio", palette=[PALETA[0], PALETA[3]], ax=ax,
    edgecolor="white", linewidth=0.8,
)
ax.set_ylabel("Participación (%)")
ax.set_xlabel("")
ax.set_title("Participación de Mercado por Sistema")
ax.legend(title="Año")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=8, padding=2)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Índice de recuperación COVID (base 100 = enero 2020)

In [ ]:
base_ene2020 = df[df["fecha"] == "2020-01-01"].set_index("sistema")["pasajeros"]

df_idx = df.copy()
df_idx["indice"] = df_idx.apply(
    lambda r: r["pasajeros"] / base_ene2020[r["sistema"]] * 100, axis=1
)

fig, ax = plt.subplots(figsize=(14, 6))
for i, sistema in enumerate(orden):
    sub = df_idx[df_idx["sistema"] == sistema]
    ax.plot(sub["fecha"], sub["indice"], color=PALETA[i], linewidth=1.8, label=sistema)

ax.axhline(100, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvspan(pd.Timestamp("2020-03-15"), pd.Timestamp("2020-07-01"),
           alpha=0.1, color="red")
ax.set_ylabel("Índice (base 100)")
ax.set_title("Recuperación Post-COVID por Sistema de Transporte")
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## Series interactivas con rango deslizable

In [ ]:
fig_linea = px.line(
    df, x="fecha", y="pasajeros", color="sistema",
    color_discrete_sequence=PALETA,
    title="Pasajeros Mensuales por Sistema",
    labels={"fecha": "Fecha", "pasajeros": "Pasajeros", "sistema": "Sistema"},
)
fig_linea.update_layout(
    xaxis=dict(
        rangeselector=dict(buttons=[
            dict(count=6, label="6m", step="month"),
            dict(count=1, label="1a", step="year"),
            dict(step="all", label="Todo"),
        ]),
        rangeslider=dict(visible=True),
    ),
    height=500,
    hovermode="x unified",
)
fig_linea.update_traces(
    hovertemplate="%{y:,.0f} pasajeros",
)
fig_linea.show()

## Distribución actual del mercado (2024)

In [ ]:
actual = df[df["anio"] == 2024].groupby("sistema")["pasajeros"].sum().reset_index()
actual["pct"] = (actual["pasajeros"] / actual["pasajeros"].sum() * 100).round(1)

fig_sun = px.sunburst(
    actual,
    path=["sistema"],
    values="pasajeros",
    color="sistema",
    color_discrete_sequence=PALETA,
    title="Distribución de Pasajeros 2024",
)
fig_sun.update_traces(
    textinfo="label+percent entry",
    hovertemplate="%{label}<br>%{value:,.0f} pasajeros<br>%{percentEntry:.1%}",
)
fig_sun.update_layout(height=500)
fig_sun.show()

## Hallazgos principales

- El Metro Línea 1 lidera en volumen de pasajeros y mostró la recuperación más rápida post-COVID.
- El Metropolitano (BRT) mantuvo su posición como segundo sistema más usado.
- Los corredores complementarios aún no recuperan sus niveles pre-pandemia al 100%.
- La estacionalidad muestra caídas en febrero (verano) y picos en julio.